# 6.5.7 Neurosymbolic AI

Hybrid pipeline: a neural digit predictor (numpy proxy) feeds into a symbolic arithmetic solver. Compare hybrid vs pure-neural accuracy under noise.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)

RULES = {'add': lambda a,b: a+b, 'mul': lambda a,b: a*b, 'max': max, 'min': min}

def neural_digit(vec, noise, rng):
    logits = vec + rng.normal(0, noise, len(vec))
    p = np.exp(logits - logits.max()); p /= p.sum()
    return np.argmax(p)

n = 30
true_a = rng.integers(0, 10, n); true_b = rng.integers(0, 10, n)
ops = rng.choice(list(RULES), n)
true_res = [RULES[op](int(a),int(b)) for op,a,b in zip(ops,true_a,true_b)]

for noise in [0.2, 0.5, 1.0]:
    correct = 0
    for i in range(n):
        va=np.zeros(10); va[true_a[i]]=2.0; vb=np.zeros(10); vb[true_b[i]]=2.0
        da=int(neural_digit(va,noise,rng)); db=int(neural_digit(vb,noise,rng))
        if RULES[ops[i]](da,db)==true_res[i]: correct+=1
    print(f'noise={noise}: hybrid acc={correct/n:.2f}')

In [ ]:
noise_levels = [0.1, 0.2, 0.5, 1.0, 2.0]
hybrid_accs, neural_accs = [], []
for noise in noise_levels:
    hc = nc = 0
    for i in range(n):
        va=np.zeros(10); va[true_a[i]]=2.0; vb=np.zeros(10); vb[true_b[i]]=2.0
        da=int(neural_digit(va,noise,rng)); db=int(neural_digit(vb,noise,rng))
        if RULES[ops[i]](da,db)==true_res[i]: hc+=1
        if da==true_a[i] and db==true_b[i]: nc+=1
    hybrid_accs.append(hc/n); neural_accs.append(nc/n)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(noise_levels, hybrid_accs, 'o-', color='steelblue', lw=2, label='Hybrid')
ax.plot(noise_levels, neural_accs, 's--', color='tomato', lw=2, label='Neural only')
ax.set(xlabel='Noise σ', ylabel='Accuracy', title='Neurosymbolic Noise Robustness')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/neurosymbolic.png')
print('Saved neurosymbolic.png')